In [3]:
import pandas as pd
import numpy as np
df_j = pd.read_parquet('../data/jaccard_overall.parquet')
df_p = pd.read_parquet('../data/researchers_publications.parquet')
df_u = pd.read_parquet('../data/umap_positions_with_source_id.parquet')
"""
    Calcule la distance entre deux chercheurs via barycentres pondérés.
    - jaccard_fp: matrice distance journaux (ISSN/EISSN).
    - pubs_fp: publications des chercheurs.
    - umap_fp: mapping source-id → ISSN/EISSN.
    """

    
    # 2. Charger publications et mapping
    
    # mapping source-id → ISSN/EISSN
mapping = df_u.set_index('source-id')['id'].astype(str).to_dict()
def get_weights(author):
        row = df_p[df_p['author_name'] == author]
        if row.empty:
            raise KeyError(f"Auteur '{author}' non trouvé")
        journals = row.iloc[0]['journals']
        issn_counts = []
        for j in journals:
            sid = str(j['journal_id'])
            issn = mapping.get(sid)
            if issn and issn in df_j.index:
                issn_counts.append((issn, j['count']))
        if not issn_counts:
            raise ValueError(f"Aucun journal valide pour '{author}'")
        ids, counts = zip(*issn_counts)
        w = np.array(counts, dtype=float)
        w /= w.sum()
        return list(ids), w


def researcher_distance(author1, author2,
                        jaccard_fp='../data/jaccard_overall.parquet',
                        pubs_fp='../data/researchers_publications.parquet',
                        umap_fp='../data/umap_positions_with_source_id.parquet'):
    
   

    # 3. Récupérer IDs et poids
    ids1, w1 = get_weights(author1)
    ids2, w2 = get_weights(author2)

    # 4. Construire la liste unique d'ISSN et extraire sous-matrice
    combined = []
    for _id in ids1 + ids2:
        if _id not in combined:
            combined.append(_id)
    M = df_j.loc[combined, combined].astype(float).values
    D2 = M**2

    # 5. Indices relatifs pour each researcher
    a_idx = [combined.index(i) for i in ids1]
    b_idx = [combined.index(i) for i in ids2]

    # 6. Extraire sous-blocs et calculs
    D2_ab = D2[np.ix_(a_idx, b_idx)]
    D2_aa = D2[np.ix_(a_idx, a_idx)]
    D2_bb = D2[np.ix_(b_idx, b_idx)]
    term_ab = w1 @ D2_ab @ w2
    term_aa = w1 @ D2_aa @ w1
    term_bb = w2 @ D2_bb @ w2

    # 7. Distance finale
    d2 = term_ab - 0.5 * term_aa - 0.5 * term_bb
    return float(np.sqrt(max(d2, 0.0)))

# Exemple d'utilisation :
dist = researcher_distance("Giroire F.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire H.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire J.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire B.", "Nisse N.")
print(f"Distance : {dist:.3f}")

Distance : 0.185
Distance : 0.395
Distance : 0.474
Distance : 0.455


In [2]:
import pandas as pd

df = pd.read_parquet("../data/umap_positions_with_source_id.parquet")
mapping = df[["source-id", "id"]].rename(columns={"id": "issn"})
mapping.to_parquet("../data/journal_mapping.parquet", index=False)

# calcul des plus proches


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# 1. Charger les embeddings de journaux
journal_emb = pd.read_parquet("../data/journalsEmb.parquet")  # ou .jsonl si tu as
# Mettre l'ID en string si besoin
journal_emb["ID"] = journal_emb["ID"].astype(str)
journal_emb = journal_emb.set_index("ID")
emb_dim = [col for col in journal_emb.columns if col.startswith("Dim_")]
emb_mat = journal_emb[emb_dim]

# 2. Charger les infos auteurs
authors = pd.read_parquet("../data/researchers_publications.parquet")

# 3. Calculer le barycentre pondéré pour chaque auteur
def author_barycenter(row):
    vec = np.zeros(len(emb_dim), dtype=float)
    total = 0
    for entry in row['journals']:
        jid = str(entry['journal_id'])
        count = entry['count']
        if jid in journal_emb.index:
            vec += journal_emb.loc[jid, emb_dim].values * count
            total += count
    return vec / total if total > 0 else vec

# 1. on calcule le c

In [ ]:
import pandas as pd
import numpy as np
import time


journals_df = pd.read_parquet('../data/journalsEmb.parquet')
researchers_df = pd.read_parquet('../data/researchers_publications.parquet')
journal_mapping_df = pd.read_parquet('../data/journal_mapping.parquet')


sourceid_to_issn = dict(zip(journal_mapping_df['source-id'].astype(str), journal_mapping_df['issn'].astype(str)))


journal_embs = journals_df.set_index('ID').dropna().astype(float)
journal_embs.index = journal_embs.index.astype(str)  

def get_barycenter(author_journals):
    vecs = []
    counts = []
    for entry in author_journals:
        source_id = str(entry['journal_id'])
        count = entry['count']
        issn = sourceid_to_issn.get(source_id)
        if issn and issn in journal_embs.index:
            vecs.append(journal_embs.loc[issn].values)
            counts.append(count)
    if not vecs:
        return None
    vecs = np.array(vecs)
    counts = np.array(counts)
    barycenter = np.average(vecs, axis=0, weights=counts)
    return barycenter


all_barycenters = []
authors = []
for i, row in researchers_df.iterrows():
    bary = get_barycenter(row['journals'])
    if bary is not None:
        all_barycenters.append(bary)
        authors.append(row['author_name'])

all_barycenters = np.array(all_barycenters)


if len(authors) == 0:
    raise RuntimeError("Aucun barycentre calculé ")

query_idx = 0
query_author = authors[query_idx]
query_bary = all_barycenters[query_idx]


start = time.time()


dists = np.linalg.norm(all_barycenters - query_bary, axis=1)
dists[query_idx] = np.inf  


nearest_idx = np.argpartition(dists, 300)[:300]
nearest_authors = [authors[i] for i in nearest_idx]
nearest_dists = dists[nearest_idx]

end = time.time()
elapsed = end - start

print(f"Premier auteur : {query_author}")
print(f"Temps pour chercher 300 voisins les plus proches : {elapsed:.3f} secondes")
print("Exemple de voisins :", nearest_authors[:10])


n_authors = len(authors)
total_est = elapsed * n_authors
print(f"Estimation temps total pour tous les chercheurs : {total_est/60:.2f} minutes (pour {n_authors} chercheurs)")


Premier auteur : Cantalapiedra-Hijar G.
Temps pour chercher 300 voisins les plus proches : 86.106 secondes
Exemple de voisins : ['Faverdin P.', 'Kondjoyan N.', "O'Donovan M.", 'Pérez-Ramírez E.', 'Boudon A.', 'Ortigues-Marty I.', 'Ferlay A.', 'Guatteo R.', 'Fanchone A.', 'Delagarde R.']
Estimation temps total pour tous les chercheurs : 2587017.46 minutes (pour 1802674 chercheurs)


plus de details dans le temps 


In [ ]:
import pandas as pd
import numpy as np
import time

global_start = time.time()


t0 = time.time()
journals_df = pd.read_parquet('../data/journalsEmb.parquet')
t1 = time.time()
print(f"[LOG] Lecture journalsEmb.parquet : {t1-t0:.2f} sec")

t0 = time.time()
researchers_df = pd.read_parquet('../data/researchers_publications.parquet')
t1b = time.time()
print(f"[LOG] Lecture researchers_publications.parquet : {t1b-t0:.2f} sec")

t0 = time.time()
journal_mapping_df = pd.read_parquet('../data/journal_mapping.parquet')
t2 = time.time()
print(f"[LOG] Lecture journal_mapping.parquet : {t2-t0:.2f} sec")


t0 = time.time()
sourceid_to_issn = dict(zip(journal_mapping_df['source-id'].astype(str), journal_mapping_df['issn'].astype(str)))
t3 = time.time()
print(f"[LOG] Mapping source-id -> issn : {t3-t0:.2f} sec")


t0 = time.time()
journal_embs = journals_df.set_index('ID').dropna().astype(float)
journal_embs.index = journal_embs.index.astype(str)
t4 = time.time()
print(f"[LOG] Indexation issn -> vecteur : {t4-t0:.2f} sec")


def get_barycenter(author_journals):
    vecs = []
    counts = []
    for entry in author_journals:
        source_id = str(entry['journal_id'])
        count = entry['count']
        issn = sourceid_to_issn.get(source_id)
        if issn and issn in journal_embs.index:
            vecs.append(journal_embs.loc[issn].values)
            counts.append(count)
    if not vecs:
        return None
    vecs = np.array(vecs)
    counts = np.array(counts)
    barycenter = np.average(vecs, axis=0, weights=counts)
    return barycenter

t0 = time.time()
all_barycenters = []
authors = []
for i, row in enumerate(researchers_df.itertuples()):
    bary = get_barycenter(row.journals)
    if bary is not None:
        all_barycenters.append(bary)
        authors.append(row.author_name)
    if (i+1) % 100_000 == 0:
        print(f"[LOG] {i+1} chercheurs traités...")

all_barycenters = np.array(all_barycenters)
t5 = time.time()
print(f"[LOG] Calcul de tous les barycentres : {t5-t0:.2f} sec")

if len(authors) == 0:
    raise RuntimeError("Aucun barycentre calculé ! Vérifie tes mappings.")


t0 = time.time()
query_idx = 0
query_author = authors[query_idx]
query_bary = all_barycenters[query_idx]
t6 = time.time()
print(f"[LOG] Sélection d'un auteur : {t6-t0:.2f} sec")


t0 = time.time()
dists = np.linalg.norm(all_barycenters - query_bary, axis=1)
dists[query_idx] = np.inf 
nearest_idx = np.argpartition(dists, 300)[:300]
nearest_authors = [authors[i] for i in nearest_idx]
nearest_dists = dists[nearest_idx]
t7 = time.time()
print(f"[LOG] Calcul distances + voisins : {t7-t0:.2f} sec")

print(f"Premier auteur : {query_author}")
print(f"Temps pour chercher 300 voisins les plus proches : {t7-t0:.3f} secondes")
print("Exemple de voisins :", nearest_authors[:10])

n_authors = len(authors)
total_est = (t7-t0) * n_authors
print(f"Estimation temps total pour tous les chercheurs : {total_est/60:.2f} minutes (pour {n_authors} chercheurs)")

global_end = time.time()
print(f"[LOG] Temps total exécution script : {global_end-global_start:.2f} sec")


[LOG] Lecture journalsEmb.parquet : 25.99 sec
[LOG] Lecture researchers_publications.parquet : 21.84 sec
[LOG] Lecture journal_mapping.parquet : 0.53 sec
[LOG] Mapping source-id -> issn : 0.83 sec
[LOG] Indexation issn -> vecteur : 1.52 sec
[LOG] 100000 chercheurs traités...
[LOG] 200000 chercheurs traités...
[LOG] 300000 chercheurs traités...
[LOG] 400000 chercheurs traités...
[LOG] 500000 chercheurs traités...
[LOG] 600000 chercheurs traités...
[LOG] 700000 chercheurs traités...
[LOG] 800000 chercheurs traités...
[LOG] 900000 chercheurs traités...
[LOG] 1000000 chercheurs traités...
[LOG] 1100000 chercheurs traités...
[LOG] 1200000 chercheurs traités...
[LOG] 1300000 chercheurs traités...
[LOG] 1400000 chercheurs traités...
[LOG] 1500000 chercheurs traités...
[LOG] 1600000 chercheurs traités...
[LOG] 1700000 chercheurs traités...
[LOG] 1800000 chercheurs traités...
[LOG] 1900000 chercheurs traités...
[LOG] 2000000 chercheurs traités...
[LOG] 2100000 chercheurs traités...
[LOG] 220000

## on tente avec faiss 


In [ ]:
import pandas as pd
import numpy as np
import time

import faiss

global_start = time.time()

t0 = time.time()
journals_df = pd.read_parquet('../data/journalsEmb.parquet')
t1 = time.time()
print(f"[LOG] Lecture journalsEmb.parquet : {t1-t0:.2f} sec")

t0 = time.time()
researchers_df = pd.read_parquet('../data/researchers_publications.parquet')
t1b = time.time()
print(f"[LOG] Lecture researchers_publications.parquet : {t1b-t0:.2f} sec")

t0 = time.time()
journal_mapping_df = pd.read_parquet('../data/journal_mapping.parquet')
t2 = time.time()
print(f"[LOG] Lecture journal_mapping.parquet : {t2-t0:.2f} sec")

t0 = time.time()
sourceid_to_issn = dict(zip(journal_mapping_df['source-id'].astype(str), journal_mapping_df['issn'].astype(str)))
t3 = time.time()
print(f"[LOG] Mapping source-id -> issn : {t3-t0:.2f} sec")

t0 = time.time()
journal_embs = journals_df.set_index('ID').dropna().astype(float)
journal_embs.index = journal_embs.index.astype(str)
t4 = time.time()
print(f"[LOG] Indexation issn -> vecteur : {t4-t0:.2f} sec")

def get_barycenter(author_journals):
    vecs = []
    counts = []
    for entry in author_journals:
        source_id = str(entry['journal_id'])
        count = entry['count']
        issn = sourceid_to_issn.get(source_id)
        if issn and issn in journal_embs.index:
            vecs.append(journal_embs.loc[issn].values)
            counts.append(count)
    if not vecs:
        return None
    vecs = np.array(vecs)
    counts = np.array(counts)
    barycenter = np.average(vecs, axis=0, weights=counts)
    return barycenter

t0 = time.time()
all_barycenters = []
authors = []
for i, row in enumerate(researchers_df.itertuples()):
    bary = get_barycenter(row.journals)
    if bary is not None:
        all_barycenters.append(bary)
        authors.append(row.author_name)
    if (i+1) % 100_000 == 0:
        print(f"[LOG] {i+1} chercheurs traités...")

all_barycenters = np.array(all_barycenters).astype('float32')  
t5 = time.time()
print(f"[LOG] Calcul de tous les barycentres : {t5-t0:.2f} sec")

if len(authors) == 0:
    raise RuntimeError("Aucun barycentre calculé ! Vérifie tes mappings.")

t0 = time.time()
query_idx = 0
query_author = authors[query_idx]
query_bary = all_barycenters[query_idx].reshape(1, -1).astype('float32')
t6 = time.time()
print(f"[LOG] Sélection d'un auteur : {t6-t0:.2f} sec")



# ----- Utilisation de Faiss -----


d = all_barycenters.shape[1]
t0 = time.time()
index = faiss.IndexFlatL2(d)
index.add(all_barycenters)
t7 = time.time()
print(f"[LOG] Index Faiss construit + data ajoutée : {t7-t0:.2f} sec")


t0 = time.time()
D, I = index.search(query_bary, 301)  # D: distances, I: indices
t8 = time.time()
print(f"[LOG] Recherche Faiss (300 voisins) : {t8-t0:.4f} sec")

nearest_idx = I[0][1:301]  # enlève le premier (lui-même)
nearest_authors = [authors[i] for i in nearest_idx]
nearest_dists = D[0][1:301]

print(f"Premier auteur : {query_author}")
print(f"Temps pour chercher 300 voisins les plus proches : {t8-t0:.4f} secondes")
print("Exemple de voisins :", nearest_authors[:10])

global_end = time.time()
print(f"[LOG] Temps total exécution script : {global_end-global_start:.2f} sec")


[LOG] Lecture journalsEmb.parquet : 2.61 sec
[LOG] Lecture researchers_publications.parquet : 12.30 sec
[LOG] Lecture journal_mapping.parquet : 0.05 sec
[LOG] Mapping source-id -> issn : 0.05 sec
[LOG] Indexation issn -> vecteur : 0.32 sec
[LOG] 100000 chercheurs traités...
[LOG] 200000 chercheurs traités...
[LOG] 300000 chercheurs traités...
[LOG] 400000 chercheurs traités...
[LOG] 500000 chercheurs traités...
[LOG] 600000 chercheurs traités...
[LOG] 700000 chercheurs traités...
[LOG] 800000 chercheurs traités...
[LOG] 900000 chercheurs traités...
[LOG] 1000000 chercheurs traités...
[LOG] 1100000 chercheurs traités...
[LOG] 1200000 chercheurs traités...
[LOG] 1300000 chercheurs traités...
[LOG] 1400000 chercheurs traités...
[LOG] 1500000 chercheurs traités...
[LOG] 1600000 chercheurs traités...
[LOG] 1700000 chercheurs traités...
[LOG] 1800000 chercheurs traités...
[LOG] 1900000 chercheurs traités...
[LOG] 2000000 chercheurs traités...
[LOG] 2100000 chercheurs traités...
[LOG] 2200000

In [15]:
argent=0.0

for i in range(12*30):
    argent+=200
    argent*=1.0041
    mois = (i % 12) + 1         # 1 à 12
    annee = (i // 12) + 1       # 1 à 10
    argentD= (i+1)*200
    print(f"Au mois numero {mois:02d} de l'anée {annee:02d} : {argent:.2f} € (sur {argentD:02d} euros deposés )")



Au mois numero 01 de l'anée 01 : 200.82 € (sur 200 euros deposés )
Au mois numero 02 de l'anée 01 : 402.46 € (sur 400 euros deposés )
Au mois numero 03 de l'anée 01 : 604.93 € (sur 600 euros deposés )
Au mois numero 04 de l'anée 01 : 808.23 € (sur 800 euros deposés )
Au mois numero 05 de l'anée 01 : 1012.37 € (sur 1000 euros deposés )
Au mois numero 06 de l'anée 01 : 1217.34 € (sur 1200 euros deposés )
Au mois numero 07 de l'anée 01 : 1423.15 € (sur 1400 euros deposés )
Au mois numero 08 de l'anée 01 : 1629.80 € (sur 1600 euros deposés )
Au mois numero 09 de l'anée 01 : 1837.31 € (sur 1800 euros deposés )
Au mois numero 10 de l'anée 01 : 2045.66 € (sur 2000 euros deposés )
Au mois numero 11 de l'anée 01 : 2254.87 € (sur 2200 euros deposés )
Au mois numero 12 de l'anée 01 : 2464.93 € (sur 2400 euros deposés )
Au mois numero 01 de l'anée 02 : 2675.86 € (sur 2600 euros deposés )
Au mois numero 02 de l'anée 02 : 2887.65 € (sur 2800 euros deposés )
Au mois numero 03 de l'anée 02 : 3100.31 €

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

input_dir = Path("../data")  # Mets le chemin vers ton dossier data si besoin

# Chargement
print("Lecture researchers_publications.parquet...")
researchers_df = pd.read_parquet(input_dir / "researchers_publications.parquet")

print("Lecture journalsEmb.parquet...")
journals_df = pd.read_parquet(input_dir / "journalsEmb.parquet")

print("Lecture journal_mapping.parquet...")
journal_mapping_df = pd.read_parquet(input_dir / "journal_mapping.parquet")

# Mapping source-id -> issn
sourceid_to_issn = dict(zip(journal_mapping_df['source-id'].astype(str), journal_mapping_df['issn'].astype(str)))
journal_embs = journals_df.set_index('ID').dropna().astype(float)
journal_embs.index = journal_embs.index.astype(str)

def get_barycenter(author_journals):
    vecs = []
    counts = []
    for entry in author_journals:
        source_id = str(entry['journal_id'])
        count = entry['count']
        issn = sourceid_to_issn.get(source_id)
        if issn and issn in journal_embs.index:
            vecs.append(journal_embs.loc[issn].values)
            counts.append(count)
    if not vecs:
        return None
    vecs = np.array(vecs)
    counts = np.array(counts)
    barycenter = np.average(vecs, axis=0, weights=counts)
    return barycenter

print("Calcul des barycentres pour tous les chercheurs...")
barycenters = []
author_names = []

for i, row in enumerate(researchers_df.itertuples()):
    bary = get_barycenter(row.journals)
    if bary is not None:
        barycenters.append(bary)
        author_names.append(row.author_name)
    if (i+1) % 100_000 == 0:
        print(f"{i+1} chercheurs traités...")

barycenters = np.array(barycenters).astype('float32')

print(f"Sauvegarde de {len(author_names)} barycentres...")

# Option 1 : Sauvegarde Pickle (très simple à relire)
with open(input_dir / "faiss_barycenters.pkl", "wb") as f:
    pickle.dump({"barycenters": barycenters, "author_names": author_names}, f)

print("✅ Fichier faiss_barycenters.pkl créé !")

# Option 2 (si tu veux): enregistre aussi en Numpy/Parquet, je peux te filer l'exemple si besoin.


Lecture researchers_publications.parquet...
Lecture journalsEmb.parquet...
Lecture journal_mapping.parquet...
Calcul des barycentres pour tous les chercheurs...
100000 chercheurs traités...
200000 chercheurs traités...
300000 chercheurs traités...
400000 chercheurs traités...
500000 chercheurs traités...
600000 chercheurs traités...
700000 chercheurs traités...
800000 chercheurs traités...
900000 chercheurs traités...
1000000 chercheurs traités...
1100000 chercheurs traités...
1200000 chercheurs traités...
1300000 chercheurs traités...
1400000 chercheurs traités...
1500000 chercheurs traités...
1600000 chercheurs traités...
1700000 chercheurs traités...
1800000 chercheurs traités...
1900000 chercheurs traités...
2000000 chercheurs traités...
2100000 chercheurs traités...
2200000 chercheurs traités...
2300000 chercheurs traités...
2400000 chercheurs traités...
Sauvegarde de 1802674 barycentres...
✅ Fichier faiss_barycenters.pkl créé !
